# Travelling Salesman Problem (TSP)

Given a set of locations, find a closed path of minimal length that visits each location exactly once.

Modeled with successor variables and a `Circuit` constraint.

This notebook uses GPS coordinates of places in Leuven. Pick a list below (`colleges` or `churches`) and solve the TSP over it.


In [2]:
import cpmpy as cp
import numpy as np

## Distance matrix

For GPS coordinates `(lat, lon)` we use great-circle (haversine) distance in metres.


In [3]:
def compute_haversine_distance_matrix(locations):
    """Integer metre distances between (lat, lon) points."""
    locs = np.asarray(locations, dtype=float)
    R = 6371000  # Earth radius (m)
    lat = np.radians(locs[:, 0])
    lon = np.radians(locs[:, 1])
    dlat = lat[:, None] - lat[None, :]
    dlon = lon[:, None] - lon[None, :]
    a = np.sin(dlat / 2) ** 2 + np.cos(lat)[:, None] * np.cos(lat)[None, :] * np.sin(dlon / 2) ** 2
    return (2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))).round().astype(int)


## TSP model

`succ[i] = j` means location `j` is visited immediately after `i`. The `Circuit` constraint forces one Hamiltonian tour.

Objective is the minimize the total distance traveled: we minimize the sum of the costs for all edges taken (sum of `Element` constraints).


In [4]:
def tsp(locations):
    """Build a Circuit-based TSP model over GPS `locations` (lat, lon).

    Returns (model, (succ,)) with travel distance minimized.
    """
    n = len(locations)
    distance_matrix = cp.cpm_array(compute_haversine_distance_matrix(locations))

    succ = cp.intvar(0, n - 1, shape=n)
    model = cp.Model(cp.Circuit(succ))
    model.minimize(cp.sum(distance_matrix[i, succ[i]] for i in range(n)))
    return model, (succ,)


## Places in Leuven

Choose one of the lists: historic university colleges, churches/abbeys, or specialty coffee bars.


In [5]:
churches = [
    ("Sint-Pieterskerk",                     50.879542, 4.701203),  # Grote Markt
    ("Sint-Michielskerk",                    50.876272, 4.701015),  # Naamsestraat
    ("Sint-Jan-de-Doperkerk",                50.871389, 4.697477),  # Groot Begijnhof
    ("Sint-Antoniuskapel",                   50.875974, 4.698421),  # Pater Damiaanplein
    ("Sint-Geertruikerk",                    50.884100, 4.699322),  # Halfmaartstraat
    ("Sint-Kwintenskerk",                    50.870002, 4.698570),  # Naamsepoort
    ("Sint-Jacobskerk",                      50.880531, 4.691123),  # Sint-Jacobsplein
    ("Onze-Lieve-Vrouw-ten-Predikherenkerk", 50.878923, 4.696110),  # Onze-Lieve-Vrouwstraat
    ("Abdij van Park",                       50.864784, 4.717672),  # Heverlee
    ("Abdij van Vlierbeek",                  50.891779, 4.736180),  # Kessel-Lo
    ("Abdij Keizersberg",                    50.888153, 4.696617),  # Mechelsestraat
]


In [6]:
colleges = [
    # iconic university landmarks
    ("Universiteitshal",              50.877898, 4.700551),  # Krakenstraat / Oude Markt
    ("Universiteitsbibliotheek",      50.877997, 4.707320),  # Mgr. Ladeuzeplein
    # Naamsestraat spine
    ("Heilige-Geestcollege",          50.876899, 4.700013),  # Naamsestraat 40
    ("Sint-Annacollege",              50.876775, 4.700731),  # Naamsestraat 37-39
    ("Koningscollege",                50.875958, 4.701021),  # Naamsestraat 59 (Instituut Dierkunde)
    ("Premonstreitcollege",           50.875583, 4.701178),  # Naamsestraat 61
    ("Atrechtcollege",                50.875255, 4.700635),  # Naamsestraat 63
    ("Hogenheuvelcollege",            50.874673, 4.700992),  # Naamsestraat 69
    ("Van Dalecollege",               50.874890, 4.699575),  # Naamsestraat 80
    ("Amerikaans College",            50.873217, 4.698988),  # Naamsestraat 100 (ex Aulnecollege)
    # around Sint-Michiel / Hogeschoolplein
    ("Maria-Theresiacollege",         50.876434, 4.702449),  # Sint-Michielsstraat 6
    ("Veteranencollege",              50.876680, 4.701385),  # Sint-Michielsstraat 2-4
    ("Pauscollege",                   50.877039, 4.703202),  # Hogeschoolplein 3
    ("Standonckcollege",              50.877221, 4.701203),  # Hogeschoolplein 6
    # west of centre
    ("Hollands College",               50.876243, 4.696164),  # Pater Damiaanplein 9
    ("Iers College",                  50.875686, 4.696241),  # Janseniusstraat 1
    ("Justus Lipsiuscollege",         50.877114, 4.694778),  # Minderbroedersstraat 15
    ("Heilige Drievuldigheidscollege", 50.877278, 4.698954),  # Oude Markt 28
    # east of centre / Tiensestraat
    ("College De Valk",               50.877951, 4.704819),  # Tiensestraat 41 (Pedagogie De Valk)
    ("College van Luik",              50.878212, 4.702666),  # Muntstraat 9-19
    ("Sint-Ivocollege",               50.878699, 4.704757),  # Leopold Vanderkelenstraat 30 (in M)
    ("Leo XIII-seminarie",            50.875617, 4.707704),  # Charles Deberiotstraat
    # north of centre
    ("Villerscollege",                50.881302, 4.702601),  # Vaartstraat 24
    ("Luxemburgcollege",              50.881825, 4.702133),  # Vaartstraat 30-32
    ("Drietalencollege",              50.881171, 4.700056),  # Busleidengang 1 (Collegium Trilingue)
    ("Malderuscollege",               50.881001, 4.706128),  # Sint-Maartenstraat 10-12
]


In [7]:
places = colleges

names = [name for name, _, _ in places]
locations = [(lat, lon) for _, lat, lon in places]

print(f"Visiting {len(locations)} places):")
for i, name in enumerate(names):
    print(f"  {i}: {name}")


Visiting 26 places):
  0: Universiteitshal
  1: Universiteitsbibliotheek
  2: Heilige-Geestcollege
  3: Sint-Annacollege
  4: Koningscollege
  5: Premonstreitcollege
  6: Atrechtcollege
  7: Hogenheuvelcollege
  8: Van Dalecollege
  9: Amerikaans College
  10: Maria-Theresiacollege
  11: Veteranencollege
  12: Pauscollege
  13: Standonckcollege
  14: Hollands College
  15: Iers College
  16: Justus Lipsiuscollege
  17: Heilige Drievuldigheidscollege
  18: College De Valk
  19: College van Luik
  20: Sint-Ivocollege
  21: Leo XIII-seminarie
  22: Villerscollege
  23: Luxemburgcollege
  24: Drietalencollege
  25: Malderuscollege


## Solve


In [8]:
model, (succ,) = tsp(locations)

if model.solve():
    print(model.status())
    print("Tour distance (m):", int(model.objective_value()))

    s = succ.value()
    tour = [0]
    i = 0
    while s[i] != 0:
        i = s[i]
        tour.append(i)
    tour.append(0)
    print(" --> ".join(names[i] for i in tour))
else:
    raise ValueError("Model is unsatisfiable")


ExitStatus.OPTIMAL (0.052942 seconds)
Tour distance (m): 3978
Universiteitshal --> Drietalencollege --> Luxemburgcollege --> Villerscollege --> Malderuscollege --> Universiteitsbibliotheek --> Leo XIII-seminarie --> College De Valk --> Sint-Ivocollege --> College van Luik --> Pauscollege --> Maria-Theresiacollege --> Koningscollege --> Premonstreitcollege --> Atrechtcollege --> Hogenheuvelcollege --> Van Dalecollege --> Amerikaans College --> Iers College --> Hollands College --> Justus Lipsiuscollege --> Heilige Drievuldigheidscollege --> Heilige-Geestcollege --> Sint-Annacollege --> Veteranencollege --> Standonckcollege --> Universiteitshal


## Plot on a map (optional)

Requires `plotly`.


In [9]:
# ! pip install plotly --quiet
import plotly.graph_objects as go

order = tour  # from previous cell
tour_lats = [locations[i][0] for i in order]
tour_lons = [locations[i][1] for i in order]

fig = go.Figure()
fig.add_trace(go.Scattermap(
    lat=tour_lats, lon=tour_lons, mode="lines",
    line=dict(width=3, color="steelblue"), hoverinfo="skip", showlegend=False,
))
fig.add_trace(go.Scattermap(
    lat=tour_lats[:-1], lon=tour_lons[:-1], mode="markers+text",
    text=[str(k) for k in range(len(order) - 1)],
    textposition="top center",
    marker=dict(size=14, color="crimson"),
    customdata=np.array([[k, names[i]] for k, i in enumerate(order[:-1])], dtype=object),
    hovertemplate="%{customdata[0]}. %{customdata[1]}<extra></extra>",
    showlegend=False,
))
fig.update_layout(
    title=f"Shortest tour — {int(model.objective_value())} m",
    height=700, margin=dict(l=0, r=0, t=40, b=0),
    map=dict(
        style="open-street-map",
        center=dict(lat=float(np.mean(tour_lats)), lon=float(np.mean(tour_lons))),
        zoom=13,
    ),
)
fig.show()
